# Lab 04: Silver Quality Gates, Imputation & Final Schema Contract
**Course:** Business Intelligence and Big Data Analysis (`01 31 0367 00`)  
**Date & Format:** Lab #4 (EN: Thu 26.11.2026 / FR: Fri 27.11.2026) · 3 Hours (180 min)

---

## 🎯 1. Context & Architecture-as-Code Lineage
In Lab 03 you normalized keys, parsed timestamps/dates, and removed duplicates into `silver_part1/*.parquet` — while leaving corrupted sensor values and bad records intact. Today you complete the entire **Silver Layer** on **Case 0 (`EuroPack Beverage Bottling`)** one file and one small step at a time (**Problem → Discovery → Solution**):

*(Note: Multi-table Joins, Cardinality, and the `480.0 min` Shift Capacity Fan-Out Trap are tackled directly in SQL during **Lab 05: DuckDB Star Schema Modeling**!)*

```mermaid
flowchart TD
    S1["2. Load silver_part1/*.parquet (2.1)"] --> O0["3. Statistical Outliers: IQR vs 3-Sigma & Active Flagging (3.1 - 3.3)"]
    O0 --> Q0["4. DAMA Gate 1: Row Quarantine vs Sensor Nullification (4.1 - 4.4)"]
    Q0 --> I0["5. Interpolation (5.1 - 5.2) -> delta_temp_C (5.3) -> PII Gate (5.4)"]
    I0 --> C0["6. Schema Contract (6.1) -> Save Parquets (6.2) -> DuckDB Audit (6.3)"]
```

**Your Objectives:**
1. **Compare Outlier Methods & Apply IQR Flagging (Section 3):** Discover how `-99.0` / `85.0 bar` spikes distort 3-Sigma (`[-8.06, 16.26] bar`), inspect the exact `2,984` rows flagged by **IQR only** (real `0.0 bar` stoppages & `5.7–8.4 bar` valve stress — NOT garbage to delete!), and apply Tukey IQR non-destructively via `is_pressure_iqr_outlier`.
2. **Two-Tier DAMA Quality Gate 1 (Section 4):** Separate **unrecoverable row errors** (`DQR-101, 102, 104, 105` — 11 rows quarantined & removed) from **recoverable single-sensor glitches** (`DQR-103` — 40 bad-pressure rows carrying `34,395` real bottles: audit-logged, nullified to `NaN`, and kept for interpolation!).
3. **Time-Series Interpolation, Feature Engineering & Privacy PII Gate (Section 5):** Repair `pasteurizer_temp_C` (`120` gaps) and nullified `filler_pressure_bar` (`40` spikes) via per-line `.interpolate(method='time')` (saving `142,704` real bottles!), derive `delta_temp_C`, and verify zero PII columns across all 5 Silver tables using `set.isdisjoint()`.
4. **Final Silver Schema Contract & PyArrow/DuckDB Audit (Section 6):** Enforce explicit `SILVER_SCHEMAS` + UTC audit timestamps, save `silver/*.parquet`, and audit Bronze vs. Final Silver Arrow schemas, Parquet footer stats, and DuckDB `DESCRIBE`.

## ⚙️ 2. Environment Setup & Silver Part 1 Input Verification
### 2.1 Load All 5 `silver_part1` Tables & Verify Starting Baseline

In [ ]:
from pathlib import Path

_cwd = Path.cwd().resolve()
REPO_ROOT = None
for _cand in [_cwd, *_cwd.parents]:
    if (_cand / "03_labs_student_starters").exists():
        REPO_ROOT = _cand
        break
    for _sub in ("ife_bi_bigdata_course", "ife_bi_bigdata_student"):
        if (_cand / _sub / "03_labs_student_starters").exists():
            REPO_ROOT = _cand / _sub
            break
    if REPO_ROOT is not None:
        break
assert REPO_ROOT is not None, f"Could not locate 03_labs_student_starters from {_cwd}"

CASE_NAME = "case0_europack_bottling"
STUDENT_BRONZE_DIR = REPO_ROOT / "03_labs_student_starters" / "lab_02_bronze_ingestion_and_profiling" / "data" / CASE_NAME
REF_CASE_DATA = REPO_ROOT / "05_case_studies_reference_solutions" / CASE_NAME / "data"
BRONZE_RAW = STUDENT_BRONZE_DIR if STUDENT_BRONZE_DIR.exists() else (REF_CASE_DATA / "bronze")
SANDBOX_DATA = REPO_ROOT / "03_labs_student_starters" / "sandbox_outputs" / CASE_NAME


def resolve_stage_dir(stage_name: str) -> Path:
    """Read upstream stage from student sandbox_outputs if populated, else fallback read-only to reference solution."""
    cand = SANDBOX_DATA / stage_name
    if cand.exists() and any(cand.glob("*.parquet")):
        return cand
    ref_cand = REF_CASE_DATA / stage_name
    if ref_cand.exists():
        return ref_cand
    raise FileNotFoundError(
        f"Stage '{stage_name}' not found in {cand} (and no reference fallback exists). Run the prerequisite Lab notebook first."
    )

import duckdb
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

BRONZE_STAGED = resolve_stage_dir("bronze_staged")
P1_DIR = resolve_stage_dir("silver_part1")
SILVER_OUT = SANDBOX_DATA / "silver"
SILVER_OUT.mkdir(parents=True, exist_ok=True)

df_telem = pd.read_parquet(P1_DIR / "silver_part1_telemetry.parquet")
df_stop = pd.read_parquet(P1_DIR / "silver_part1_stoppages.parquet")
df_lines = pd.read_parquet(P1_DIR / "silver_lines.parquet")
df_shifts = pd.read_parquet(P1_DIR / "silver_shifts.parquet")
df_formats = pd.read_parquet(P1_DIR / "silver_bottle_formats.parquet")

print("Silver Part 1 source :", P1_DIR)
print("Silver Final output  :", SILVER_OUT)
print(
    f"Loaded 5 Silver Part 1 tables -> "
    f"telemetry: {len(df_telem):,} rows (timestamp={df_telem['timestamp'].dtype}) | "
    f"stoppages: {len(df_stop):,} logs | lines: {len(df_lines)} | shifts: {len(df_shifts)} | formats: {len(df_formats)}"
)


## 📊 3. Statistical Outlier Analysis — Tukey IQR vs. Gaussian 3-Sigma (`df_telem`)
### 3.1 Discovery — Inspect Telemetry Distributions & Extreme Spikes
**The Problem:** Before choosing any outlier rule, inspect `filler_pressure_bar` and `pasteurizer_temp_C` summary statistics (`mean`, `std`, `min`, `50%`, `max`). Notice how just 40 extreme sensor spikes (`-99.0` and `85.0 bar`) inflate `std` to `4.05 bar`!

> **Pandas Syntax Reference:**
> | Method | Purpose | Example |
> | :--- | :--- | :--- |
> | `df[cols].describe()` | Summary stats (`count`, `mean`, `std`, `min`, `25%`, `50%`, `75%`, `max`) | `df_telem[["filler_pressure_bar", "pasteurizer_temp_C"]].describe()` |
> | `df.loc[mask, cols]` | Filter rows matching a boolean condition | `df_telem.loc[(df_telem["filler_pressure_bar"] < 0) \| (df_telem["filler_pressure_bar"] > 12)]` |

In [ ]:
print("=== Summary Statistics (notice how -99.0 / 85.0 bar inflate filler_pressure_bar std!) ===")
print(df_telem[["filler_pressure_bar", "pasteurizer_temp_C"]].describe().round(2).to_string())

extreme_p = df_telem.loc[(df_telem["filler_pressure_bar"] < 0.0) | (df_telem["filler_pressure_bar"] > 12.0)]
print(f"\nExtreme pressure readings outside physical [0.0, 12.0] bar: {len(extreme_p)} rows")
print(extreme_p[["timestamp", "line_id", "bottle_format", "filler_pressure_bar", "bottled_units"]].head(5).to_string(index=False))


### 3.2 Solution — Compare Tukey IQR vs. 3-Sigma & Inspect Exactly Which Rows Differ
**The Insight:** Implement both estimators and inspect the **disagreement set** (rows flagged by IQR only vs. rows flagged by 3-Sigma only):
- **3-Sigma (`mean ± 3*std`):** Widens to `[-8.06, 16.26] bar` — so distorted that it accepts impossible negative pressures and catches *only* the 40 extreme spikes (`0` rows flagged by 3-Sigma only!).
- **Tukey IQR (`Q1 - 1.5*IQR` .. `Q3 + 1.5*IQR`):** Tightly anchored at `[3.10, 5.42] bar` — flags `3,024` rows (`2,984` rows missed by 3-Sigma!).
- **Why you must INSPECT before deleting:** Those `2,984` IQR-only rows are **`0.00 bar` real line stoppages** (`bottled_units = 0`) and **`5.7–8.4 bar` real valve-wear stress** right before breakdowns! That is why industrial pipelines use **Hard Physics Bounds (`[0, 12] bar`)** for sensor faults and **IQR** for *flagging* statistical stress (never blindly dropping!).

> **Pandas Syntax Reference:**
> | Method | Purpose | Example |
> | :--- | :--- | :--- |
> | `Series.dropna().quantile(q)` | Compute Q1 (`0.25`) and Q3 (`0.75`) | `q1, q3 = valid.quantile(0.25), valid.quantile(0.75)` |
> | `Series.mean()` / `Series.std()` | Sample mean and standard deviation | `mu, sd = valid.mean(), valid.std()` |
> | `mask_iqr & ~mask_s3` | Rows flagged by IQR **only** (missed by 3-Sigma) | `df_telem.loc[mask_iqr & ~mask_s3]` |

In [ ]:
def get_iqr_bounds(s: pd.Series, k: float = 1.5):
    # TODO: Drop NaNs, compute Q1=quantile(0.25), Q3=quantile(0.75), iqr=Q3-Q1, return (float(q1 - k*iqr), float(q3 + k*iqr))
    return None

def get_3sigma_bounds(s: pd.Series):
    # TODO: Drop NaNs, compute mu=mean(), sd=std(), return (float(mu - 3*sd), float(mu + 3*sd))
    return None

if get_iqr_bounds(df_telem["filler_pressure_bar"]) is not None and get_3sigma_bounds(df_telem["filler_pressure_bar"]) is not None:
    rows = []
    for col in ("filler_pressure_bar", "pasteurizer_temp_C"):
        iqr_lo, iqr_hi = get_iqr_bounds(df_telem[col])
        s3_lo, s3_hi = get_3sigma_bounds(df_telem[col])
        m_iqr = (df_telem[col] < iqr_lo) | (df_telem[col] > iqr_hi)
        m_s3 = (df_telem[col] < s3_lo) | (df_telem[col] > s3_hi)
        rows.append({
            "column": col,
            "IQR_bounds": f"[{iqr_lo:.2f}, {iqr_hi:.2f}]",
            "IQR_flagged": int(m_iqr.sum()),
            "3Sigma_bounds": f"[{s3_lo:.2f}, {s3_hi:.2f}]",
            "3Sigma_flagged": int(m_s3.sum()),
            "IQR_only": int((m_iqr & ~m_s3).sum()),
            "3Sigma_only": int((m_s3 & ~m_iqr).sum()),
        })
    print("=== IQR vs 3-SIGMA COMPARISON (RAW TELEMETRY) ===")
    print(pd.DataFrame(rows).to_string(index=False))

    p_iqr_lo, p_iqr_hi = get_iqr_bounds(df_telem["filler_pressure_bar"])
    p_s3_lo, p_s3_hi = get_3sigma_bounds(df_telem["filler_pressure_bar"])
    iqr_only_p = df_telem.loc[
        ((df_telem["filler_pressure_bar"] < p_iqr_lo) | (df_telem["filler_pressure_bar"] > p_iqr_hi))
        & df_telem["filler_pressure_bar"].between(p_s3_lo, p_s3_hi)
    ]
    print(f"\nSample of the {len(iqr_only_p):,} pressure rows flagged by IQR ONLY (real 0.00 bar stoppages & 5.7-8.4 bar stress!):")
    print(iqr_only_p[["timestamp", "line_id", "bottle_format", "filler_pressure_bar", "bottled_units"]].head(6).to_string(index=False))
else:
    print("⏳ [TODO] Implement get_iqr_bounds() and get_3sigma_bounds() above.")


### 3.3 Solution — Put Tukey IQR to Work: Flag Active-Production Pressure Anomalies
**The Practical Use of IQR:** Once we exclude idle stoppage rows (`bottled_units == 0`) and impossible sensor glitches (outside `[0, 12] bar`), computing Tukey IQR on **active running production** gives a tight normal operating band (`[3.22, 5.38] bar`). Instead of deleting stressed rows, tag them with a boolean column **`is_pressure_iqr_outlier`** so Maintenance teams can analyze pressure excursions in Gold without losing a single bottle!

> **Pandas Syntax Reference:**
> | Method | Purpose | Example |
> | :--- | :--- | :--- |
> | `Series.between(lo, hi)` | Inclusive range check (`lo <= x <= hi`) | `df_telem["filler_pressure_bar"].between(0.0, 12.0)` |
> | `active_mask & ~s.between(lo, hi)` | Flag statistical outliers only on active valid rows | `df_telem["is_pressure_iqr_outlier"] = active_valid & ~df_telem["filler_pressure_bar"].between(act_lo, act_hi)` |

In [ ]:
active_valid = df_telem["filler_pressure_bar"].between(0.0, 12.0) & (df_telem["bottled_units"] > 0)

if get_iqr_bounds(df_telem["filler_pressure_bar"]) is not None and get_3sigma_bounds(df_telem["filler_pressure_bar"]) is not None:
    act_iqr_lo, act_iqr_hi = get_iqr_bounds(df_telem.loc[active_valid, "filler_pressure_bar"])
    act_s3_lo, act_s3_hi = get_3sigma_bounds(df_telem.loc[active_valid, "filler_pressure_bar"])

    # TODO: Create boolean column df_telem["is_pressure_iqr_outlier"] = active_valid & ~df_telem["filler_pressure_bar"].between(act_iqr_lo, act_iqr_hi)
    if "is_pressure_iqr_outlier" in df_telem.columns:
        n_iqr_act = int(df_telem["is_pressure_iqr_outlier"].sum())
        n_s3_act = int((active_valid & ~df_telem["filler_pressure_bar"].between(act_s3_lo, act_s3_hi)).sum())
        print(f"Active running pressure bounds -> IQR: [{act_iqr_lo:.2f}, {act_iqr_hi:.2f}] ({n_iqr_act:,} flagged) | 3-Sigma: [{act_s3_lo:.2f}, {act_s3_hi:.2f}] ({n_s3_act:,} flagged)")
        print("Tagged `is_pressure_iqr_outlier` without deleting any valid production rows!")
    else:
        print("⏳ [TODO] Assign df_telem['is_pressure_iqr_outlier'] above.")
else:
    print("⏳ [TODO] Complete Step 3.2 first.")


## 🚨 4. DAMA Data Quality Gate 1 — Row Quarantine vs. Sensor-Glitch Nullification
Not all data errors deserve the same action! In production pipelines we separate two tiers:
1. **Unrecoverable row-level errors (`QUARANTINED_ROW`):** Unknown asset (`LINE_Z`), impossible date (`2099`), or corrupted production counts (`rejected_units > bottled_units`). These rows **must be removed** from Silver and archived in `quarantine_errors.parquet`.
2. **Recoverable single-sensor glitches (`NULLIFIED_AND_INTERPOLATED`):** A pressure sensor reading `-99.0` or `85.0 bar` on a row with valid `bottled_units`. We log it, nullify only `filler_pressure_bar`, and interpolate in Section 5.

### 4.1 Discovery (File 1: `df_telem`) — Flag Unrecoverable Row Errors (`DQR-101, 102, 104`)
**The Problem:** Identify the three unrecoverable telemetry error classes (read-only first so we can inspect what will be quarantined):
- `DQR-101` (`Consistency`): Orphan `line_id` not in ERP `valid_lines` (`LINE_Z`)
- `DQR-102` (`Timeliness`): Null (`NaT`) or future `timestamp > 2027-01-01`
- `DQR-104` (`Validity`): Negative units or `rejected_units > bottled_units`

> **Pandas Syntax Reference:**
> | Method / Operator | Purpose | Example |
> | :--- | :--- | :--- |
> | `~df["line_id"].isin(valid_lines)` | Flag keys absent from the ERP master set | `q_orphan = ~df_telem["line_id"].isin(valid_lines)` |
> | `pd.Timestamp("2027-01-01")` | Native datetime threshold for timestamp comparisons | `df_telem["timestamp"] > pd.Timestamp("2027-01-01")` |
> | `cond_a \| cond_b` | Element-wise logical OR across boolean masks | `(df_telem["bottled_units"] < 0) \| (df_telem["rejected_units"] > df_telem["bottled_units"])` |

In [ ]:
valid_lines = set(df_lines["line_id"])

# TODO (Discovery — Read-Only): Build boolean masks for DQR-101, DQR-102, and DQR-104
q_orphan = None  # ~df_telem["line_id"].isin(valid_lines)
q_future = None  # df_telem["timestamp"].isna() | (df_telem["timestamp"] > pd.Timestamp("2027-01-01"))
q_units  = None  # (df_telem["bottled_units"] < 0) | (df_telem["rejected_units"] < 0) | (df_telem["rejected_units"] > df_telem["bottled_units"])

if all(m is not None for m in (q_orphan, q_future, q_units)):
    print(f"[Discovery] Unrecoverable telemetry rows -> DQR-101: {q_orphan.sum()} | DQR-102: {q_future.sum()} | DQR-104: {q_units.sum()}")
    print("\nPreview of flagged unrecoverable rows:")
    print(df_telem.loc[q_orphan | q_future | q_units, ["timestamp", "line_id", "bottle_format", "bottled_units", "rejected_units"]].to_string(index=False))
else:
    print("⏳ [TODO] Define boolean masks q_orphan, q_future, and q_units above.")


### 4.2 Solution (File 1: `df_telem`) — Quarantine & Remove Unrecoverable Rows
**The Fix:** Package the flagged rows using mutually exclusive priority masks (`q_future & ~q_orphan`, etc. so a row violating two rules is never double-counted) and remove only those unrecoverable rows from `df_telem`.

> **Pandas Syntax Reference:**
> | Method / Operator | Purpose | Example |
> | :--- | :--- | :--- |
> | `mask_b & ~mask_a` | Mutually exclusive mask (prevents double-quarantining) | `q_future & ~q_orphan` |
> | `df.loc[~bad_mask].copy()` | Keep only valid rows as an independent DataFrame | `df_telem = df_telem.loc[~(q_orphan \| q_future \| q_units)].copy()` |

In [ ]:
# Helper provided: packages flagged rows into a standardized DAMA Audit/Quarantine DataFrame
def extract_quarantine(df: pd.DataFrame, mask: pd.Series, src: str, rid: str, code: str, crit: str, action: str = "QUARANTINED_ROW") -> pd.DataFrame:
    cols = ["dq_rule_id", "error_code", "dama_criterion", "source_table", "action", "raw_record"]
    if not mask.any():
        return pd.DataFrame(columns=cols)
    return pd.DataFrame({
        "dq_rule_id": rid,
        "error_code": code,
        "dama_criterion": crit,
        "source_table": src,
        "action": action,
        "raw_record": df.loc[mask].astype(str).to_dict(orient="records"),
    }).assign(raw_record=lambda d: d["raw_record"].astype(str))

q_telem_parts = []
if all(m is not None for m in (q_orphan, q_future, q_units)):
    q_telem_parts = [
        extract_quarantine(df_telem, q_orphan, "line_telemetry.csv", "DQR-101", "ORPHAN_LINE_ID", "Consistency", "QUARANTINED_ROW"),
        extract_quarantine(df_telem, q_future & ~q_orphan, "line_telemetry.csv", "DQR-102", "INVALID_OR_FUTURE_TIMESTAMP", "Timeliness", "QUARANTINED_ROW"),
        extract_quarantine(df_telem, q_units & ~q_orphan & ~q_future, "line_telemetry.csv", "DQR-104", "NEGATIVE_OR_IMPOSSIBLE_UNITS", "Validity", "QUARANTINED_ROW"),
    ]
    before_n = len(df_telem)
    # TODO: Filter out unrecoverable rows -> df_telem = df_telem.loc[~(q_orphan | q_future | q_units)].copy()
    print(f"[Verification] Removed {before_n - len(df_telem)} unrecoverable rows -> df_telem: {len(df_telem):,} rows.")
else:
    print("⏳ [TODO] Complete Step 4.1 masks first.")


### 4.3 Discovery & Solution (File 1: `df_telem`) — Nullify Pressure Sensor Glitches (`DQR-103`, Keep All `34,395` Bottles!)
**Why `[0.0, 12.0] bar` are the physical bounds (domain spec from Case 0):**
- **`0.0 bar` lower bound:** Gauge pressure inside a bottling filler bowl cannot go negative — when a line is stopped/depressurized, it legitimately reads **`0.0 bar`**. Negative readings (`-99.0 bar`) are physically impossible sensor fault codes.
- **`12.0 bar` upper bound:** Normal carbonated bottling runs `3.7–5.2 bar` and stressed valves climb to `5.9–8.39 bar` right before breakdowns. Above **`12.0 bar`**, PET bottles burst and safety relief valves vent — so `85.0 bar` is guaranteed to be a corrupted sensor reading.

**Why Nullify Instead of Deleting the Row?** Those `40` bad-pressure rows carry **`34,395` real bottled units**! Log them in our audit log (`action="NULLIFIED_AND_INTERPOLATED"`) and set **only** `filler_pressure_bar = np.nan` so Section 5 repairs the pressure reading without erasing real factory production.

> **Pandas Syntax Reference:**
> | Method | Purpose | Example |
> | :--- | :--- | :--- |
> | `df.loc[mask, "col"].sum()` | Quantify production at risk before deciding an action | `df_telem.loc[q_press, "bottled_units"].sum()` |
> | `df.loc[mask, "col"] = np.nan` | Nullify ONLY the corrupted sensor column (keep the row!) | `df_telem.loc[q_press, "filler_pressure_bar"] = np.nan` |

In [ ]:
# STEP A (Discovery): Flag readings outside physical envelope [0.0, 12.0] bar
# TODO: Build q_press = (df_telem["filler_pressure_bar"] < 0.0) | (df_telem["filler_pressure_bar"] > 12.0)
q_press = None
q_press_part = pd.DataFrame()

if q_press is not None:
    press_bottles_saved = int(df_telem.loc[q_press, "bottled_units"].sum())
    print(f"[Discovery] Found {q_press.sum()} impossible pressure readings carrying {press_bottles_saved:,} real bottled_units!")
    print(df_telem.loc[q_press, ["timestamp", "line_id", "bottle_format", "filler_pressure_bar", "bottled_units"]].head(4).to_string(index=False))

    # STEP B (Solution): Log in audit table AND nullify only filler_pressure_bar (keep the rows!)
    q_press_part = extract_quarantine(df_telem, q_press, "line_telemetry.csv", "DQR-103", "PRESSURE_OUT_OF_BOUNDS_0_12", "Accuracy", "NULLIFIED_AND_INTERPOLATED")
    # TODO: df_telem.loc[q_press, "filler_pressure_bar"] = np.nan
    print(f"\n[Verification] Null filler_pressure_bar count ready for Section 5 interpolation: {df_telem['filler_pressure_bar'].isna().sum()}")
else:
    print("⏳ [TODO] Define q_press and nullify filler_pressure_bar above.")


### 4.4 Discovery & Solution (File 2: `df_stop`) — Quarantine `DQR-105` & Assemble `quarantine_df`
**The Problem:** Apply the quarantine gate to File 2 (`df_stop`, isolating phantom log `STOP-20269999`), then concatenate all telemetry and stoppage audit slices into `quarantine_df` (`11` `QUARANTINED_ROW` + `40` `NULLIFIED_AND_INTERPOLATED` = **`51` audited DQ events**).

> **Pandas Syntax Reference:**
> | Method | Purpose | Example |
> | :--- | :--- | :--- |
> | `pd.concat(list_of_dfs, ignore_index=True)` | Stack DataFrames vertically with clean index `0..N-1` | `quarantine_df = pd.concat([p for p in q_parts if not p.empty], ignore_index=True)` |
> | `df.groupby(["dq_rule_id", "error_code", "action"]).size()` | Audit counts grouped by rule and action taken | `quarantine_df.groupby(["dq_rule_id", "error_code", "action"]).size()` |

In [ ]:
# STEP A (Discovery): Flag invalid stoppage logs in df_stop
# TODO: stop_bad = (~df_stop["line_code"].isin(valid_lines)) | df_stop["event_date"].isna() | (df_stop["event_date"] > pd.Timestamp("2027-01-01"))
stop_bad = None

quarantine_df = None
if stop_bad is not None and q_telem_parts and not q_press_part.empty:
    print(f"[Discovery] Flagged {stop_bad.sum()} invalid stoppage log(s):")
    print(df_stop.loc[stop_bad, ["log_id", "event_date", "line_code", "stoppage_type", "root_cause", "downtime_min"]].to_string(index=False))

    # STEP B (Solution): Quarantine DQR-105, filter df_stop, and combine all audit slices
    q_stop_part = extract_quarantine(df_stop, stop_bad, "stoppage_logs.json", "DQR-105", "INVALID_STOPPAGE_LOG", "Completeness", "QUARANTINED_ROW")
    df_stop = df_stop.loc[~stop_bad].copy()

    q_parts = [*q_telem_parts, q_press_part, q_stop_part]
    quarantine_df = pd.concat([p for p in q_parts if not p.empty], ignore_index=True)

    print("\n[Solution] DAMA DQ Audit Summary (Quarantined Rows + Nullified Sensor Glitches):")
    print(quarantine_df.groupby(["dq_rule_id", "error_code", "action"]).size().to_string())
    print(f"Total audited DQ events: {len(quarantine_df)} | Clean df_stop rows: {len(df_stop):,}")
else:
    print("⏳ [TODO] Complete Steps 4.1–4.3 and define stop_bad above.")


## 🌡️ 5. Per-Line Time Interpolation, Feature Engineering & Privacy PII Gate
### 5.1 Discovery — Locate All Sensor Gaps (`pasteurizer_temp_C` & Nullified `filler_pressure_bar`)
**The Problem:** `df_telem` now has **two** recoverable sensor gap sets on valid production rows:
1. **`120` missing `pasteurizer_temp_C` readings** (`108,309` real bottled units!)
2. **`40` nullified `filler_pressure_bar` spikes** (`34,395` real bottled units preserved in Step 4.3!)

Sort chronologically per `line_id` and inspect both gap counts alongside neighboring 15-minute readings.

> **Pandas Syntax Reference:**
> | Method | Purpose | Example |
> | :--- | :--- | :--- |
> | `df.sort_values(["line_id", "timestamp"]).reset_index(drop=True)` | Sort chronologically per line before time interpolation | Required before `.interpolate(method="time")` |
> | `Series.isna()` | Boolean mask of missing (`NaN`) cells | `missing_temp = df_telem["pasteurizer_temp_C"].isna()` |
> | `df.loc[mask, "bottled_units"].sum()` | Total real production saved by interpolating instead of dropping | `df_telem.loc[missing_temp \| missing_press, "bottled_units"].sum()` |

In [ ]:
df_telem = df_telem.sort_values(["line_id", "timestamp"]).reset_index(drop=True)
missing_temp = df_telem["pasteurizer_temp_C"].isna()
missing_press = df_telem["filler_pressure_bar"].isna()
total_saved_bottles = int(df_telem.loc[missing_temp | missing_press, "bottled_units"].sum())

print(f"Missing pasteurizer_temp_C gaps : {missing_temp.sum():>3} ({int(df_telem.loc[missing_temp, 'bottled_units'].sum()):,} bottles)")
print(f"Nullified filler_pressure_bar   : {missing_press.sum():>3} ({int(df_telem.loc[missing_press, 'bottled_units'].sum()):,} bottles)")
print(f"Total real bottles preserved    : {total_saved_bottles:,} bottles (saved vs dropna()!)")

first_temp_idx = int(missing_temp.idxmax())
print("\nSample pasteurizer_temp_C gap with adjacent 15-min readings:")
print(df_telem.loc[max(0, first_temp_idx - 1):first_temp_idx + 1, ["timestamp", "line_id", "pasteurizer_temp_C", "bottled_units"]].to_string(index=False))

if missing_press.any():
    first_press_idx = int(missing_press.idxmax())
    print("\nSample nullified filler_pressure_bar gap with adjacent 15-min readings:")
    print(df_telem.loc[max(0, first_press_idx - 1):first_press_idx + 1, ["timestamp", "line_id", "filler_pressure_bar", "bottled_units"]].to_string(index=False))
else:
    first_press_idx = None


### 5.2 Solution — Per-Line Time-Series Interpolation (`method='time'`)
**The Fix:** Record every repaired row in **`is_imputed_flag = missing_temp | missing_press`**, then interpolate BOTH `pasteurizer_temp_C` and `filler_pressure_bar` linearly over time (**`method="time"`**) **independently per `line_id`** so readings never bleed across two different machines!

> **Pandas Syntax Reference:**
> | Method | Purpose | Example |
> | :--- | :--- | :--- |
> | `g.set_index("timestamp")[col].interpolate(method="time", limit_direction="both")` | Time-weighted linear interpolation on a `DatetimeIndex` | Inside `_interp_col(g, col)` per `line_id` |
> | `Series.fillna(interp_vals)` | Replace only `NaN` cells with aligned interpolated values | `df_telem[col] = df_telem[col].fillna(interp_vals).round(2)` |
> | `Series.between(0.0, 12.0).all()` | Assert 100% of repaired pressures lie inside `[0.0, 12.0]` bar | `assert df_telem["filler_pressure_bar"].between(0.0, 12.0).all()` |

In [ ]:
df_telem["is_imputed_flag"] = missing_temp | missing_press

def _interp_col(g: pd.DataFrame, col: str) -> pd.Series:
    s = g.set_index("timestamp")[col].interpolate(method="time", limit_direction="both")
    return pd.Series(s.values, index=g.index)

# TODO: Interpolate both "pasteurizer_temp_C" and "filler_pressure_bar" per line_id:
# for col in ("pasteurizer_temp_C", "filler_pressure_bar"):
#     interp_vals = pd.concat([_interp_col(grp, col) for _, grp in df_telem.groupby("line_id", sort=False)]).sort_index()
#     df_telem[col] = df_telem[col].fillna(interp_vals).round(2)

if df_telem["pasteurizer_temp_C"].isna().sum() == 0 and df_telem["filler_pressure_bar"].isna().sum() == 0:
    assert df_telem["filler_pressure_bar"].between(0.0, 12.0).all(), "Out-of-bounds pressure remains!"
    print("Temperature gap AFTER per-line time interpolation:")
    print(df_telem.loc[max(0, first_temp_idx - 1):first_temp_idx + 1, ["timestamp", "line_id", "pasteurizer_temp_C", "is_imputed_flag"]].to_string(index=False))
    if first_press_idx is not None:
        print("\nPressure spike AFTER nullify + per-line time interpolation:")
        print(df_telem.loc[max(0, first_press_idx - 1):first_press_idx + 1, ["timestamp", "line_id", "filler_pressure_bar", "is_imputed_flag", "bottled_units"]].to_string(index=False))
    print(f"\n[OK] Imputed {int(df_telem['is_imputed_flag'].sum())} sensor gaps ({missing_temp.sum()} temp + {missing_press.sum()} pressure) -> 0 nulls!")
else:
    print("⏳ [TODO] Interpolate pasteurizer_temp_C and filler_pressure_bar per line_id above.")


### 5.3 Solution — Feature Engineering (`delta_temp_C`)
**The Goal:** Now that `pasteurizer_temp_C` has zero missing values, compute the thermal differential **`delta_temp_C = pasteurizer_temp_C - ambient_temp_C`**. Maintenance teams use `delta_temp_C` alongside `valve_cycles_k` in Lab 05 to spot overheating before breakdowns!

> **Pandas Syntax Reference:**
> | Method | Purpose | Example |
> | :--- | :--- | :--- |
> | `(df["col_a"] - df["col_b"]).round(2)` | Vectorized element-wise subtraction rounded to 2 decimals | `df_telem["delta_temp_C"] = (df_telem["pasteurizer_temp_C"] - df_telem["ambient_temp_C"]).round(2)` |

In [ ]:
# TODO: Compute df_telem["delta_temp_C"] = (df_telem["pasteurizer_temp_C"] - df_telem["ambient_temp_C"]).round(2)

if "delta_temp_C" in df_telem.columns and df_telem["delta_temp_C"].isna().sum() == 0:
    print("Derived delta_temp_C summary statistics:")
    print(df_telem["delta_temp_C"].describe().round(2).to_string())
else:
    print("⏳ [TODO] Complete Step 5.2 and compute df_telem['delta_temp_C'] above.")


### 5.4 Verification — Privacy / PII Exclusion Gate (`set.isdisjoint`)
**The Problem:** Under GDPR and corporate data governance, **Personally Identifiable Information (PII)** — such as employee names, emails, national IDs, or medical notes — must **never** leak into analytical Silver or Gold tables. Scan the column names of **all 5 Silver tables** against `PII_FORBIDDEN_COLS` before persisting anything to disk!

> **Python `set` Syntax Reference:**
> | Method | Purpose | Example |
> | :--- | :--- | :--- |
> | `set_a.isdisjoint(set_b)` | Returns **`True`** if `set_a` and `set_b` share **zero elements in common** (empty intersection); **`False`** if any forbidden column exists | `PII_FORBIDDEN_COLS.isdisjoint({c.lower() for c in df.columns})` |
> | `set_a & set_b` | Set intersection — returns the exact leaking column names (useful for error messages!) | `leaks = PII_FORBIDDEN_COLS & {c.lower() for c in df.columns}` |

In [ ]:
PII_FORBIDDEN_COLS = {
    "first_name", "last_name", "full_name",
    "corporate_email", "national_id_masked", "details_medical_note",
}

silver_tables = {
    "silver_telemetry": df_telem,
    "silver_stoppages": df_stop,
    "silver_lines": df_lines,
    "silver_shifts": df_shifts,
    "silver_bottle_formats": df_formats,
}

for tbl_name, df_tbl in silver_tables.items():
    col_set = {c.lower() for c in df_tbl.columns}
    # set.isdisjoint() returns True ONLY when col_set shares zero columns with PII_FORBIDDEN_COLS
    assert PII_FORBIDDEN_COLS.isdisjoint(col_set), f"PII leakage in {tbl_name}: {PII_FORBIDDEN_COLS & col_set}"
    print(f"  [PASS] {tbl_name:<24} -> 0 PII columns detected ({len(col_set)} columns checked)")


## 💾 6. Final Silver Schema Contract, Persistence & PyArrow / DuckDB Audit
### 6.1 Solution — Parse UTC Audit Timestamps & Enforce Explicit `SILVER_SCHEMAS`
**The Problem:** Before saving to disk, parse `_ingestion_timestamp_utc` into timezone-aware **`timestamp[ns, tz=UTC]`** across all 5 tables and cast every text, numeric, and boolean column to its explicit Silver contract (`string`, `float64`, `int64`, `bool`).

> **Pandas Syntax Reference:**
> | Method | Purpose | Example |
> | :--- | :--- | :--- |
> | `pd.to_datetime(s, format="ISO8601", utc=True, errors="coerce")` | Convert UTC audit strings -> `datetime64[ns, UTC]` | `_df["_ingestion_timestamp_utc"] = pd.to_datetime(...)` |
> | `Series.astype("string")` / `Series.astype("bool")` | Explicit nullable string & boolean dtypes | `out[col] = out[col].astype("string")` |
> | `pd.to_numeric(s, errors="coerce").astype("float64" / "int64")` | Strict numeric schema casting | `out[col] = pd.to_numeric(out[col], errors="coerce").astype("float64")` |

In [ ]:
SILVER_SCHEMAS = {
    "telemetry": {
        "string": ["line_id", "bottle_format", "shift_code", "_source_file"],
        "float64": ["ambient_temp_C", "pasteurizer_temp_C", "filler_pressure_bar", "delta_temp_C"],
        "int64": ["conveyor_speed_bpm", "valve_cycles_k", "bottled_units", "rejected_units"],
        "bool": ["is_imputed_flag", "is_pressure_iqr_outlier"],
    },
    "stoppages": {
        "string": ["log_id", "line_code", "stoppage_type", "root_cause", "technician_team", "shift_code", "_source_file"],
        "int64": ["downtime_min"],
    },
    "lines": {
        "string": ["line_id", "line_name", "filler_type", "_source_file"],
        "float64": ["nominal_cycle_time_sec", "hourly_operating_cost_eur"],
    },
    "shifts": {
        "string": ["shift_id", "shift_name", "start_time", "end_time", "_source_file"],
        "int64": ["planned_duration_hours"],
    },
    "bottle_formats": {
        "string": ["format_code", "format_name", "packaging_material", "_source_file"],
        "float64": ["unit_reject_cost_eur"],
        "int64": ["target_hourly_output"],
    },
}

def enforce_dtypes(df: pd.DataFrame, spec: dict) -> pd.DataFrame:
    out = df.copy()
    for col in spec.get("string", []):
        out[col] = out[col].astype("string")
    for col in spec.get("float64", []):
        out[col] = pd.to_numeric(out[col], errors="coerce").astype("float64")
    for col in spec.get("int64", []):
        out[col] = pd.to_numeric(out[col], errors="coerce").astype("int64")
    for col in spec.get("bool", []):
        out[col] = out[col].astype("bool")
    return out

schema_ready = (
    quarantine_df is not None
    and "delta_temp_C" in df_telem.columns
    and "is_pressure_iqr_outlier" in df_telem.columns
    and df_telem["pasteurizer_temp_C"].isna().sum() == 0
    and df_telem["filler_pressure_bar"].isna().sum() == 0
)

if schema_ready:
    for _df in (df_telem, df_stop, df_lines, df_shifts, df_formats):
        _df["_ingestion_timestamp_utc"] = pd.to_datetime(
            _df["_ingestion_timestamp_utc"], format="ISO8601", utc=True, errors="coerce"
        )
    df_telem = enforce_dtypes(df_telem, SILVER_SCHEMAS["telemetry"])
    df_stop = enforce_dtypes(df_stop, SILVER_SCHEMAS["stoppages"])
    df_lines = enforce_dtypes(df_lines, SILVER_SCHEMAS["lines"])
    df_shifts = enforce_dtypes(df_shifts, SILVER_SCHEMAS["shifts"])
    df_formats = enforce_dtypes(df_formats, SILVER_SCHEMAS["bottle_formats"])
    print("Enforced Silver dtypes on df_telem:")
    print(df_telem.dtypes.to_string())
else:
    print("⏳ [TODO] Complete Sections 3, 4, and 5 first before enforcing Silver schemas.")


### 6.2 Solution — Save All 6 Final Silver Parquets (`silver/*.parquet`)
**The Goal:** Persist all 5 typed Silver tables plus `quarantine_errors.parquet` into `SILVER_OUT` using Snappy compression and `index=False` (preventing `__index_level_0__` leaks).

> **Pandas Syntax Reference:**
> | Method | Purpose | Example |
> | :--- | :--- | :--- |
> | `df.to_parquet(path, index=False, compression="snappy")` | Write Snappy Parquet cleanly without saving the Pandas index | `df_out.to_parquet(SILVER_OUT / fname, index=False, compression="snappy")` |

In [ ]:
silver_outputs = {}
if schema_ready:
    silver_outputs = {
        "silver_telemetry.parquet": df_telem,
        "silver_stoppages.parquet": df_stop,
        "silver_lines.parquet": df_lines,
        "silver_shifts.parquet": df_shifts,
        "silver_bottle_formats.parquet": df_formats,
    }
    for fname, df_out in silver_outputs.items():
        df_out.to_parquet(SILVER_OUT / fname, index=False, compression="snappy")
        print(f"  [SAVED] {fname:<32} ({len(df_out):>6,} rows)")
    quarantine_df.to_parquet(SILVER_OUT / "quarantine_errors.parquet", index=False, compression="snappy")
    print(f"  [SAVED] {'quarantine_errors.parquet':<32} ({len(quarantine_df):>6,} audited DQ events)")
else:
    print("⏳ [TODO] Complete Step 6.1 before saving the final Silver Parquets.")


### 6.3 Audit — Bronze vs. Final Silver Parquet Schemas, Footer Stats & DuckDB `DESCRIBE`
**The Final Sign-Off:** Compare the embedded Arrow schema, Parquet `physical_type`, and footer `null_count / min / max` bounds between raw Bronze and Final Silver — proving directly from Parquet metadata that all nulls are gone, pressures sit inside `[0.0, 12.0] bar`, `LINE_Z` & `2099` are gone, and DuckDB sees native `TIMESTAMP_NS`, `TIMESTAMPTZ_NS`, `DATE`, and `BOOLEAN` columns!

In [ ]:
if silver_outputs:
    pf_bronze = pq.ParquetFile(BRONZE_STAGED / "bronze_telemetry.parquet")
    pf_silver = pq.ParquetFile(SILVER_OUT / "silver_telemetry.parquet")
    bronze_types = {f.name: str(f.type) for f in pf_bronze.schema_arrow}
    silver_types = {f.name: str(f.type) for f in pf_silver.schema_arrow}
    rg_s = pf_silver.metadata.row_group(0)
    silver_stats = {rg_s.column(i).path_in_schema: rg_s.column(i) for i in range(rg_s.num_columns)}

    comparison = pd.DataFrame([
        {
            "column": col,
            "bronze_type": bronze_types.get(col, "<new>"),
            "silver_type": silver_types[col],
            "phys_type": silver_stats[col].physical_type,
            "nulls": silver_stats[col].statistics.null_count if silver_stats[col].statistics else None,
            "footer_min": repr(silver_stats[col].statistics.min) if silver_stats[col].statistics else None,
            "footer_max": repr(silver_stats[col].statistics.max) if silver_stats[col].statistics else None,
        }
        for col in silver_types
    ])
    print("=== BRONZE vs FINAL SILVER PARQUET SCHEMA & FOOTER STATS ===")
    print(comparison.to_string(index=False))

    # Assert strict Arrow/Parquet temporal & date contracts across ALL 5 saved Silver tables
    for fname in silver_outputs:
        sc = pq.ParquetFile(SILVER_OUT / fname).schema_arrow
        assert "__index_level_0__" not in set(sc.names), f"Index leak in {fname}!"
        ing_t = sc.field("_ingestion_timestamp_utc").type
        assert pa.types.is_timestamp(ing_t) and ing_t.tz == "UTC", f"{fname}: expected UTC timestamp, got {ing_t}"

    assert pa.types.is_timestamp(pf_silver.schema_arrow.field("timestamp").type)
    assert pa.types.is_date(pf_silver.schema_arrow.field("operational_shift_date").type)
    stop_sc = pq.ParquetFile(SILVER_OUT / "silver_stoppages.parquet").schema_arrow
    assert pa.types.is_timestamp(stop_sc.field("event_date").type)
    assert pa.types.is_date(stop_sc.field("operational_shift_date").type)

    # Verify DuckDB sees native TIMESTAMP_NS / TIMESTAMPTZ_NS / DATE / BOOLEAN types directly from Parquet
    con = duckdb.connect()
    duck_schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{SILVER_OUT / 'silver_telemetry.parquet'}')").df()
    con.close()
    print("\nDuckDB DESCRIBE on silver_telemetry.parquet:")
    print(duck_schema[["column_name", "column_type"]].to_string(index=False))
    print(f"\n[PASS] Complete Silver Layer ready for Lab 05 (DuckDB Star Schema & Joins) -> {SILVER_OUT}")
else:
    print("⏳ [TODO] Run Steps 6.1 and 6.2 first to audit the final Silver Parquets.")
